In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/darshanv/credit-risk-scorer/data/application_train.csv')
print(f"Shape: {df.shape}")

Shape: (307511, 122)


In [2]:
# 365243 is a placeholder for unemployed people
# If we leave it, the model thinks they've been 
# employed for 1000 years — completely wrong
print("Before fix:", df['DAYS_EMPLOYED'].max())
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
print("After fix:", df['DAYS_EMPLOYED'].max())

Before fix: 365243
After fix: 0.0


In [3]:
# Features you proved were useful during EDA
df['age_years'] = -df['DAYS_BIRTH'] / 365
df['income_annuity_ratio'] = df['AMT_INCOME_TOTAL'] / df['AMT_ANNUITY']
df['employment_to_age_ratio'] = (-df['DAYS_EMPLOYED']) / (-df['DAYS_BIRTH'])
df['null_count'] = df.isnull().sum(axis=1)
df['credit_to_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

print("New features created successfully")
print(df[['age_years', 'income_annuity_ratio', 
          'employment_to_age_ratio',
          'null_count', 
          'credit_to_income_ratio']].describe())

New features created successfully
           age_years  income_annuity_ratio  employment_to_age_ratio  \
count  307511.000000         307499.000000            252137.000000   
mean       43.936973              7.352846                 0.156861   
std        11.956133              9.441929                 0.133549   
min        20.517808              0.533059                -0.000000   
25%        34.008219              4.365541                 0.056099   
50%        43.150685              6.141249                 0.118733   
75%        53.923288              8.712181                 0.219170   
max        69.120548           4466.586497                 0.728811   

          null_count  credit_to_income_ratio  
count  307511.000000           307511.000000  
mean       30.123231                3.957570  
std        20.959126                2.689728  
min         0.000000                0.004808  
25%         6.000000                2.018667  
50%        37.000000                3.265067

/var/folders/qf/k3qfl89s3hz2ym7jfndgvf300000gn/T/ipykernel_23903/383573457.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_years'] = -df['DAYS_BIRTH'] / 365
/var/folders/qf/k3qfl89s3hz2ym7jfndgvf300000gn/T/ipykernel_23903/383573457.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['income_annuity_ratio'] = df['AMT_INCOME_TOTAL'] / df['AMT_ANNUITY']
/var/folders/qf/k3qfl89s3hz2ym7jfndgvf300000gn/T/ipykernel_23903/383573457.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of ca

In [4]:
# Anything above 40% missing gets dropped
# We proved during EDA these aren't reliable
threshold = 0.4
missing_frac = df.isnull().sum() / len(df)
cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()

# But NEVER drop EXT_SOURCE columns even if >40% missing
cols_to_drop = [c for c in cols_to_drop 
                if 'EXT_SOURCE' not in c]

df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")
print(f"Remaining columns: {df.shape[1]}")

Dropped 48 columns
Remaining columns: 79


In [5]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Fill numeric missing with median
# Median is better than mean because it ignores outliers
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing with mode (most common value)
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after imputation:")
print(df.isnull().sum().sum())
# Should print 0

Missing values after imputation:
0


/var/folders/qf/k3qfl89s3hz2ym7jfndgvf300000gn/T/ipykernel_23903/670440056.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [6]:
# Models can't read text like "Male" or "Secondary"
# We convert them to numbers using Label Encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Categorical columns encoded")
print(f"Final shape: {df.shape}")

Categorical columns encoded
Final shape: (307511, 79)


In [7]:
# Save so you don't have to redo all this every time
df.to_csv('/Users/darshanv/credit-risk-scorer/data/application_train_processed.csv', 
          index=False)
print("Processed data saved!")
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Processed data saved!
Final dataset: 307511 rows, 79 columns
